### Agrupamento de categorias raras em `natureza_juridica`

**Entrada:** `data/dataset_modelagem_final.parquet` (45.876 linhas, 44 colunas) -- já sem o artefato EIRELI, tratado no notebook `02`.

**Saída:** o mesmo arquivo, sobrescrito -- 45 colunas no total, com `natureza_juridica_agrupada` adicionada e nenhuma linha removida ou adicionada.

#### 1. Motivo do agrupamento

6 categorias de `natureza_juridica` têm `n` muito pequeno (1 a 11, 22 casos no total) e taxa de fracasso 0% só por acaso amostral, não por viés estrutural como a EIRELI. Agrupamos essas 6 na categoria `"Outras"` em vez de remover as linhas, já que a observação é válida -- só o rótulo é raro demais para ter coeficiente próprio. Os códigos e a contagem de cada um estão no comentário da célula de código abaixo.

#### 2. Carregamento e recodificação

Carrega o checkpoint do notebook 02 e agrupa os 6 códigos raros de `natureza_juridica` na categoria `"Outras"` (seção 1).

In [1]:
import pandas as pd

# mesmo arquivo é lido e depois sobrescrito -- este notebook só adiciona uma coluna, não recria o dataset
CAMINHO = "data/dataset_modelagem_final.parquet"

df = pd.read_parquet(CAMINHO)
print("Carregado:", df.shape)

Carregado: (45876, 44)


In [2]:
# Agrupa 6 códigos raros de natureza_juridica em "Outras" -- não remove nenhuma observação, só o rótulo muda.
# No notebook 02, essas 6 categorias ficaram com taxa de fracasso 0% só por célula pequena (22 casos no
# total), não por viés estrutural como a EIRELI (nenhuma foi extinta por lei ou reclassificada):
#   2143 Cooperativa                          n=11
#   2240 Sociedade Simples Limitada           n=5
#   3999 Associação Privada                   n=2
#   2216 Empresa Domiciliada no Exterior      n=2
#   2038 Sociedade de Economia Mista          n=1
#   2127 Sociedade em Conta de Participação   n=1
# Os códigos com n grande o suficiente para taxa estável (2135, 2062, 2054, 2046) permanecem como estão.
CODIGOS_OUTRAS = ["2143", "2240", "3999", "2216", "2038", "2127"]

df["natureza_juridica_agrupada"] = df["natureza_juridica"].where(
    ~df["natureza_juridica"].isin(CODIGOS_OUTRAS), "Outras"
)

print("Valores unicos pos-recodificacao:", sorted(df["natureza_juridica_agrupada"].unique().tolist()))

Valores unicos pos-recodificacao: ['2046', '2054', '2062', '2135', 'Outras']


#### 3. Confirmação e salvamento

Confere que a categoria `"Outras"` tem taxa de fracasso não degenerada (nem 0% nem 100%) e `n` suficiente para ser estimável, e então salva o resultado sobrescrevendo o checkpoint.

In [3]:
# monta n e taxa de fracasso por categoria agrupada, para inspecionar visualmente se "Outras" ficou razoável
crosstab_n = pd.crosstab(df["natureza_juridica_agrupada"], df["alvo_baixada"])
crosstab_pct = pd.crosstab(df["natureza_juridica_agrupada"], df["alvo_baixada"], normalize="index") * 100

resumo = crosstab_n.copy()
resumo.columns = ["n_sobreviveu_0", "n_fracassou_1"]
resumo["n_total"] = resumo["n_sobreviveu_0"] + resumo["n_fracassou_1"]
resumo["taxa_fracasso_pct"] = crosstab_pct[1].round(1)
resumo = resumo.sort_values("n_total", ascending=False)
print(resumo.to_string())

# trava de sanidade: o agrupamento só recodifica rótulo, não pode ter alterado a contagem de linhas
assert len(df) == 45876, f"Esperado 45876, obtido {len(df)}"
print("\nAssert OK: ainda 45876 linhas (nenhuma removida)")

# trava de sanidade: se "Outras" ainda fosse 0% ou 100%, o agrupamento não teria resolvido o problema de separação
taxa_outras = resumo.loc["Outras", "taxa_fracasso_pct"]
assert 0 < taxa_outras < 100, "Categoria 'Outras' ainda degenerada"
print(f"Assert OK: taxa de 'Outras' nao degenerada (nem 0% nem 100%): {taxa_outras}%")

                            n_sobreviveu_0  n_fracassou_1  n_total  taxa_fracasso_pct
natureza_juridica_agrupada                                                           
2135                                 13099          24115    37214               64.8
2062                                  5523           2920     8443               34.6
2054                                    76             23       99               23.2
2046                                    83             15       98               15.3
Outras                                  17              5       22               22.7

Assert OK: ainda 45876 linhas (nenhuma removida)
Assert OK: taxa de 'Outras' nao degenerada (nem 0% nem 100%): 22.7%


In [4]:
# sobrescreve o mesmo arquivo: dataset final agora tem a coluna natureza_juridica_agrupada
df.to_parquet(CAMINHO, index=False)
print(f"Salvo (sobrescrito) em {CAMINHO}")
print("Shape final:", df.shape)

Salvo (sobrescrito) em data/dataset_modelagem_final.parquet
Shape final: (45876, 45)


#### 4. Resultado

Todos os 5 níveis de `natureza_juridica_agrupada` agora têm `n` e taxa de fracasso estimáveis, sem nenhuma categoria degenerada. A coluna original `natureza_juridica` foi mantida no dataset.